In [ ]:
import gradio as gr
import requests
import json
import logging

# ============================================
# 0. Azure Functions HTTP 엔드포인트 설정
#    - 로컬 실행: http://localhost:7071/api/chat_rag
#    - 나중에 Azure에 배포하면 → 실제 Functions URL로 변경
# ============================================
CHAT_RAG_ENDPOINT = "http://localhost:7071/api/chat_rag"


# ============================================
# 1. 백엔드 호출 함수 (Gradio → Azure Function)
# ============================================
def rag_chat(message, history):
    """
    Gradio에서 사용자의 메시지를 받아,
    Azure Functions의 chat_rag HTTP 트리거로 전달하고
    응답(answer)을 받아 챗봇에 표시하는 함수.
    """
    try:
        # Azure Function으로 POST 요청
        response = requests.post(
            CHAT_RAG_ENDPOINT,
            json={"question": message},
            timeout=30,
        )

        # HTTP 응답 코드 체크
        if response.status_code == 200:
            result = response.json()
            # {"answer": "..."} 형태를 기대
            return result.get(
                "answer",
                "죄송합니다. 데이터베이스에서 관련 정보를 찾지 못했습니다.",
            )
        else:
            return (
                "❌ Azure Function 호출 중 오류가 발생했습니다.\n"
                f"- HTTP 상태 코드: {response.status_code}\n"
                f"- 응답 내용: {response.text}"
            )

    except requests.exceptions.ConnectionError:
        return (
            "❌ Azure Function이 로컬에서 실행 중이 아닌 것 같습니다.\n"
            "- VS Code에서 함수 앱을 실행했는지 확인해주세요. (F5)\n"
            "- 또는 chat_rag 엔드포인트 주소를 다시 확인해주세요."
        )
    except Exception as e:
        logging.error(f"알 수 없는 오류: {str(e)}")
        return f"❌ 알 수 없는 오류가 발생했습니다: {str(e)}"


# ============================================
# 2. 반도체 테마 설정 (Soft 테마 + 커스텀 CSS)
# ============================================

# Gradio 기본 Soft 테마 사용 (버전 호환 안전)
semi_theme = gr.themes.Soft()

# 추가 CSS로 반도체 느낌의 다크모드 UI 적용
semi_css = """
/* 전체 배경: 반도체 웨이퍼 느낌의 그라데이션 + 그리드 라인 */
.gradio-container {
    background: radial-gradient(circle at top, #0b1120 0, #020617 40%, #000000 100%);
    color: #e5e7eb;
    font-family: system-ui, -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
}

/* 상단 타이틀 / 설명 텍스트 꾸미기 */
.gradio-container h1 {
    text-align: center;
    font-weight: 800;
    letter-spacing: 0.08em;
    color: #e0f2fe;
    text-shadow: 0 0 16px rgba(56, 189, 248, 0.7);
    margin-top: 0.8rem;
    margin-bottom: 0.4rem;
}

.gradio-container p {
    text-align: center;
    color: #94a3b8;
    margin-bottom: 0.8rem;
}

/* 상단에 반도체 칩/배선 느낌 그리드 */
.gradio-container::before {
    content: "";
    position: fixed;
    inset: 0;
    pointer-events: none;
    background-image:
        linear-gradient(to right, rgba(56,189,248,0.2) 1px, transparent 1px),
        linear-gradient(to bottom, rgba(30,64,175,0.18) 1px, transparent 1px);
    background-size: 120px 120px;
    mix-blend-mode: soft-light;
    opacity: 0.35;
    z-index: -1;
}

/* Chatbot 영역을 카드처럼 */
.chatbot {
    border-radius: 18px !important;
    box-shadow:
        0 0 0 1px rgba(148, 163, 184, 0.15),
        0 18px 40px rgba(15, 23, 42, 0.9);
    background: linear-gradient(135deg, rgba(15,23,42,0.96), rgba(15,23,42,1));
    border: 1px solid rgba(148, 163, 184, 0.28);
}

/* 채팅창 높이 조정 */
.chatbot > .wrap {
    max-height: 420px;
}

/* 유저 말풍선: 오른쪽, 네온 하이라이트 */
.chatbot .message.user {
    background: linear-gradient(135deg, #22d3ee, #6366f1) !important;
    color: #0b1120 !important;
    border-radius: 18px 18px 4px 18px !important;
    box-shadow: 0 0 18px rgba(56, 189, 248, 0.45);
    border: none !important;
}

/* 봇 말풍선: 왼쪽, 딥 네이비 + 시안 틴트 */
.chatbot .message.bot {
    background: radial-gradient(circle at top left, rgba(56,189,248,0.12), rgba(15,23,42,0.95)) !important;
    border-radius: 18px 18px 18px 4px !important;
    border: 1px solid rgba(148,163,184,0.4) !important;
}

/* 입력 텍스트 박스 */
textarea, .gr-textbox textarea {
    background: rgba(15,23,42,0.96) !important;
    border-radius: 14px !important;
    border: 1px solid rgba(148,163,184,0.55) !important;
    color: #e5e7eb !important;
}

/* 텍스트 박스 포커스 시 네온 효과 */
textarea:focus, .gr-textbox textarea:focus {
    outline: none !important;
    box-shadow: 0 0 0 1px rgba(56,189,248,0.8), 0 0 18px rgba(56,189,248,0.55);
    border-color: rgba(56,189,248,0.9) !important;
}

/* 버튼 스타일 (전송/클리어 등) */
button {
    border-radius: 999px !important;
    font-weight: 600 !important;
    letter-spacing: 0.03em;
}

/* 버튼 기본 색상 살짝 조정 */
button.primary {
    box-shadow: 0 0 14px rgba(56,189,248,0.45);
}

/* Examples(예시 질문) 버튼 스타일 */
.examples button {
    border-radius: 999px !important;
    background: rgba(15,23,42,0.9) !important;
    border: 1px solid rgba(148,163,184,0.5) !important;
    color: #e5e7eb !important;
    font-size: 0.85rem !important;
}
.examples button:hover {
    border-color: rgba(56,189,248,0.85) !important;
    color: #e0f2fe !important;
}

/* 하단 푸터 영역 숨기고 싶으면 (선택) */
/*
footer {
    display: none !important;
}
*/
"""


# ============================================
# 3. Gradio ChatInterface 구성
# ============================================
demo = gr.ChatInterface(
    fn=rag_chat,
    title="반도체 공정 이상탐지 RAG 시스템",
    description="⚙️ Cosmos DB에 저장된 벡터 기반 불량 데이터를 이용해 C-Line / H-Line 등 공정 이상 상황을 탐지하고 설명합니다.",
    chatbot=gr.Chatbot(
        height=380,
        label="📡 공정 이상 탐지 로그",
    ),
    theme=semi_theme,
    css=semi_css,
    examples=[
        "C-Line에서 최근 발생한 불량 웨이퍼 사례를 설명해줘",
        "최근 저장된 불량 웨이퍼 중에서 불량 확률이 가장 높은 케이스는 뭐야?",
        "웨이퍼 96_C-Line과 비슷한 다른 불량 사례가 있는지 알려줘",
    ],
)


# ============================================
# 4. 실행 엔트리 포인트
# ============================================
if __name__ == "__main__":
    # server_port를 지정하지 않으면, Gradio가 비어있는 포트를 자동으로 선택
    demo.launch(
        server_name="0.0.0.0",  # 같은 네트워크(와이파이) 안 다른 장비에서도 접속 가능
        share=True,             # 🌍 퍼블릭 gradio.live URL 자동 발급
    )


C:\Users\LuxClarus\AppData\Local\Temp\ipykernel_35760\2061107621.py:192: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot=gr.Chatbot(
c:\Users\LuxClarus\Desktop\azure-secom-rag\.venv\Lib\site-packages\gradio\chat_interface.py:330: UserWarning: The gr.ChatInterface was not provided with a type, so the type of the gr.Chatbot, 'tuples', will be used.
  warnings.warn(
ERROR:    [Errno 10048] error while attempting to bind on address ('0.0.0.0', 7863): 각 소켓 주소(프로토콜/네트워크 주소/포트)는 하나만 사용할 수 있습니다
ERROR:    [Errno 10048] error while attempting to bind on address ('0.0.0.0', 7864): 각 소켓 주소(프로토콜/네트워크 주소/포트)는 하나만 사용할 수 있습니다
ERROR:    [Errno 10048] error while attempting to bind on address ('0.0.0.0', 7865): 각 소켓 주소(프로토콜/네트워크 주소/포트)는 하나만 

* Running on local URL:  http://0.0.0.0:7866

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.
